In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

In [ ]:
# Load the base model first
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

# Wrap it with your adapters
ver_model = PeftModel.from_pretrained(base_model, "/kaggle/input/datasets/ahmedamrabolfadl/verified-labeler/outputs/checkpoint-813")
ver_tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/ahmedamrabolfadl/verified-labeler/outputs/checkpoint-813")

tax_model = PeftModel.from_pretrained(base_model, "/kaggle/input/datasets/ahmedamrabolfadl/taxonomy-labeler/outputs/checkpoint-6250")
tax_tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/ahmedamrabolfadl/taxonomy-labeler/outputs/checkpoint-6250")

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/testing_pairs.csv')
df.dropna()
df.head()

In [ ]:
import torch

SYSTEM_PROMPT = """You are an expert at analyzing research topics and generating concise, descriptive labels.
Given a set of keywords and a research field, generate a clear and relevant topic label that captures the essence of the topic.
Provide only the topic label without any additional explanation."""

max_seq_length=1024

def infer_topic_label(topic_words, research_field, model, tokenizer):
    prompt_text = f"System Prompt: {SYSTEM_PROMPT}\n\nTopic Keywords: {topic_words}\nResearch Field: {research_field}\nAnswer:"
    
    # Tokenize
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=max_seq_length)
    
    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.0
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the first answer after the last "Answer:" in the prompt
    try:
        # Split on "Answer:" and take the last part
        label = generated_text.split("Answer:")[-1]
        # Sometimes the model writes multiple lines, take only the first line
        label = label.strip().split("\n")[0]
    except:
        label = generated_text.strip()

    return label

In [ ]:
# Apply inference to test_df to create ver_label column
df['ver_label'] = df.apply(
    lambda row: infer_topic_label(row['topic_words'], row['research_field'], ver_model, ver_tokenizer),
    axis=1
)

df['tax_label'] = df.apply(
    lambda row: infer_topic_label(row['topic_words'], row['research_field'], tax_model, tax_tokenizer),
    axis=1
)

# Display results
df.head()

In [ ]:
df.to_csv('/kaggle/working/qwen_labels.csv', index=False)